In [1]:
#packages
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
import math as m
from pyspark.ml.stat import Correlation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

25/10/05 13:20:39 WARN Utils: Your hostname, Parths-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.22 instead (on interface en0)
25/10/05 13:20:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/05 13:20:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#reading in data\n

tbl_merchants_raw = spark.read.parquet('../data/tables/merchant_data/tbl_merchants.parquet')
consumer_user_details = spark.read.parquet('../data/tables/merchant_data/consumer_user_details.parquet')
transactions21 = spark.read.parquet('../data/tables/transaction_data/transactions_20210228_20210827_snapshot/')
transactions2122 = spark.read.parquet('../data/tables/transaction_data/transactions_20210828_20220227_snapshot/')
transactions22 = spark.read.parquet('../data/tables/transaction_data/transactions_20220228_20220828_snapshot/')
con_fraud_prob = spark.read.option("header","true").csv('../data/tables/merchant_data/consumer_fraud_probability.csv')
merch_fraud_prob = spark.read.option("header", "true").csv('../data/tables/merchant_data/merchant_fraud_probability.csv')

tbl_consumer_raw = spark.read.option("header", "true").csv('../data/tables/merchant_data/tbl_consumer.csv')

transactions = transactions21.unionByName(transactions2122)
transactions = transactions.unionByName(transactions22)

In [4]:
#Functions
def OHE_variables(data, cat_nom_columns, cat_ord_columns):
    
    """Indexes and encodes categorical features"""

    for c in cat_nom_columns:
        indexer = StringIndexer(inputCol=c, outputCol=str(c) + '_index')
        
        indexed_df = indexer.fit(data).transform(data)
        data.drop(c)
        encoder = OneHotEncoder(inputCol=str(c)+'_index', outputCol=str(c)+'_OHE')
        encoded_df = encoder.fit(indexed_df).transform(indexed_df)
        data.drop(str(c)+'_index')

    for c in cat_ord_columns:
        indexer = StringIndexer(inputCol=c, outputCol=str(c) + '_index')
        indexed_df = indexer.fit(data).transform(data)
        data.drop(c)
        
    return data

def find_NULL(dfs):
    for df in dfs:
        condition = f.lit(False)
        for col_name in df.columns:
            condition = condition | f.col(col_name).isNull()

        df.filter(condition).show()
    return df.filter(condition).count()

def filter_outliers(data, variables):
    
    """filters outliers of continuous data"""

    n=data.count()
    for feature in variables:
        # Calculate Q1 and Q3
        quantiles = data.approxQuantile(feature, [0.25, 0.75], 0.01)
        q1, q3 = quantiles
        iqr = q3 - q1

        #from ADS lecture slides, n>>100
        scale = m.sqrt(m.log(n)) - 0.5
        if scale<3:
            scale=3
        lower_bound = q1 - scale * iqr
        upper_bound = q3 + scale * iqr
        if lower_bound<0:
            data = data.filter((col(feature) >= 0) & (col(feature) <= upper_bound))
        else:
            data = data.filter((col(feature) >= lower_bound) & (col(feature) <= upper_bound))
    
    return data

def corr_func(data, CORR_COLS):

    """A function to return the correlation matrix of correlation between variables"""

    features = "correlation_features"

    assembler = VectorAssembler(
        inputCols=CORR_COLS, 
        outputCol=features 
    )
    
    feature_vector = assembler.transform(data).select(features)

    corr_matrix_dense = Correlation.corr(feature_vector, features)
    corr_matrix_dense.collect()
    corr_matrix = corr_matrix_dense.collect()[0][0].toArray().tolist()

    return corr_matrix

def spark_shape(self):
        return (self.count(), len(self.columns))
pyspark.sql.dataframe.DataFrame.shape = property(spark_shape)

In [5]:
#cleaning tags
string = "name|address|state|postcode|gender|consumer_id"

# Clean consumer table
tbl_consumer = (
    tbl_consumer_raw
    .withColumn("cust_name", f.split(col(string), "\\|").getItem(0))
    .withColumn("address", f.split(col(string), "\\|").getItem(1))
    .withColumn("state", f.split(col(string), "\\|").getItem(2))
    .withColumn("postcode", f.split(col(string), "\\|").getItem(3))
    .withColumn("gender", f.split(col(string), "\\|").getItem(4))
    .withColumn("consumer_id", f.split(col(string), "\\|").getItem(5))
    .drop(string)
)


# Clean merchants table
tbl_merchants = (
    tbl_merchants_raw
    # remove leading (( or [[ and trailing )) or ]]
    .withColumn(
        "tags_clean",
        f.regexp_replace(
            "tags",
            r"^\(\(|^\(\[|^\[\(|^\[\[|\)\)$|\]\)$|\)\]$|\]\]$",
            ""
        )
    )
    # split on `), (` or `], [`
    .withColumn("tags_array", f.split("tags_clean", r"\)\s*,\s*\(|\]\s*,\s*\["))
    # extract each element
    .withColumn("biz_tags", f.lower(f.col("tags_array")[0]))
    .withColumn("rev_band", f.col("tags_array")[1])
    .withColumn("take_rate", f.regexp_extract(f.col("tags_array")[2], r"take rate:\s*([0-9.]+)", 1)
    )
    .drop("tags", "tags_clean", "tags_array")
)

tbl_merchants=tbl_merchants.withColumn("biz_tags", f.regexp_replace("biz_tags", "  ", " "))

In [6]:
merchant_transactions=transactions.join(tbl_merchants, on='merchant_abn', how='left')
#find_NULL([merchant_transactions])

In [7]:
merchant_transactions = merchant_transactions.dropna()

In [8]:
#merchant_transactions.groupBy('name').count().orderBy("count", ascending=True).show()

In [9]:
#filter outliers by biz_tag

n = merchant_transactions.count()
# compute the scale factor
scale = m.sqrt(m.log(n)) - 0.5
stats_by_band = (merchant_transactions.groupby('biz_tags')
                                      .agg(f.expr("percentile_approx(dollar_value, 0.25)").alias("Q1"),
                                           f.expr("percentile_approx(dollar_value, 0.75)").alias("Q3")
                ).withColumn("IQR", f.col("Q3") - f.col("Q1"))
                 .withColumn("lower_bound", f.col("Q1") - scale * f.col("IQR"))
                 .withColumn("upper_bound", f.col("Q3") + scale * f.col("IQR"))
                )
stats_by_band = stats_by_band.drop('IQR')

In [10]:
print(merchant_transactions.shape)

(13614675, 9)


In [11]:
merchant_transactions = (
    merchant_transactions
    .join(stats_by_band, on="biz_tags", how="left")
    .filter(
        (col("dollar_value") >= col('lower_bound')) &
        (col("dollar_value") <= col("upper_bound"))
    )
    .select(merchant_transactions["*"])
)

In [12]:
print(merchant_transactions.shape)

(13293842, 9)


In [13]:
merchant_transactions=merchant_transactions.withColumnRenamed('name', 'business')
merchant_transactions=merchant_transactions.drop('order_id',)

In [14]:
merchant_transactions

merchant_abn,user_id,dollar_value,order_datetime,business,biz_tags,rev_band,take_rate
15549624934,2,130.3505283105634,2021-08-20,Commodo Associates,"opticians, optica...",c,2.76
46804135891,18482,6.6168976971833615,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
11237511112,15,86.43306925925785,2021-08-20,Magna Institute,"opticians, optica...",c,2.11
48534649627,37,115.71011998439714,2021-08-20,Dignissim Maecena...,"opticians, optica...",a,6.64
22059270846,18563,13.552320313774313,2021-08-20,Montes Nascetur R...,"opticians, optica...",a,6.59
46804135891,83,40.88736687931136,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
81410315303,90,97.01139408070111,2021-08-20,Sed Dictum PC,"opticians, optica...",a,6.35
46804135891,96,71.58993600692456,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
60602272553,112,17.429130408212,2021-08-20,Sagittis Duis Gra...,"opticians, optica...",b,4.93
46804135891,118,14.417591756971596,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93


In [15]:
merchant_transactions.groupBy("biz_tags").agg(
    f.min("dollar_value").alias("min_value"),
    f.max("dollar_value").alias("max_value"),
    f.mean("dollar_value").alias("mean"),
    (f.max("dollar_value") - f.min("dollar_value")).alias("range")
).orderBy("mean", ascending=False).show()

25/10/05 13:21:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:21:59 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------+--------------------+------------------+------------------+------------------+
|            biz_tags|           min_value|         max_value|              mean|             range|
+--------------------+--------------------+------------------+------------------+------------------+
|jewelry, watch, c...|   3.409793978681009| 46001.13901942742| 9278.563185654202| 45997.72922544874|
|art dealers and g...|  0.4127496907944707| 10335.94618503865| 1966.235783927581|10335.533435347856|
|             telecom|  0.2931526313090789| 11606.18761084434|1735.6703991984107| 11605.89445821303|
|equipment, tool, ...|0.040595292090802974| 8813.127778854296|1261.6527064827037| 8813.087183562206|
|stationery, offic...|0.004010169952587961|2333.8271055469763| 456.9907620082106| 2333.823095377024|
|health and beauty...| 6.59812930332817E-4|1672.2945523725923|294.95533758789253| 1672.293892559662|
|motor vehicle sup...|7.092782606876731E-4| 1356.587373980561| 271.4439563467199|1356.58666

In [39]:
tbl_consumer=tbl_consumer.drop('address', 'cust_name', 'gender')
tbl_consumer.show()
tbl_consumer.write.parquet("../data/curated/tbl_consumer", mode="overwrite")

+-----+--------+-----------+
|state|postcode|consumer_id|
+-----+--------+-----------+
|   WA|    6935|    1195503|
|  NSW|    2782|     179208|
|   NT|     862|    1194530|
|  NSW|    2780|     154128|
|   WA|    6355|     712975|
|  NSW|    2033|     407340|
|  QLD|    4606|     511685|
|   WA|    6056|     448088|
|  NSW|    2482|     650435|
|  VIC|    3220|    1058499|
|  VIC|    3063|     428325|
|   WA|    6743|    1494640|
|  QLD|    4673|    1146717|
|  VIC|    3332|    1343547|
|  QLD|    4512|    1463076|
|  NSW|    2452|    1356405|
|  VIC|    3719|    1331093|
|  NSW|    1109|      80965|
|  TAS|    7276|    1226530|
|  VIC|    3234|    1390367|
+-----+--------+-----------+
only showing top 20 rows



25/10/05 14:17:25 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers
25/10/05 14:17:25 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 84.47% for 8 writers
25/10/05 14:17:25 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers


In [17]:
#merch_tran_cust=merchant_transactions.join(consumer_user_details, on='user_id', how='left')
#merch_tran_cust=merch_tran_cust.join(tbl_consumer, on='consumer_id', how='left')
#merch_tran_cust=merch_tran_cust.drop('consumer_id', 'order_id')

In [18]:
mtc_fraud1=merchant_transactions.join(merch_fraud_prob, on=['merchant_abn', 'order_datetime'], how='left')
mtc_fraud1=mtc_fraud1.withColumnRenamed('fraud_probability', 'merch_fraud_prob')
mtc_fraud=mtc_fraud1.join(con_fraud_prob, on=['user_id', 'order_datetime'], how='left')
mtc_fraud=mtc_fraud.withColumnRenamed('fraud_probability', 'con_fraud_prob')

In [19]:
mtc_fraud

user_id,order_datetime,merchant_abn,dollar_value,business,biz_tags,rev_band,take_rate,merch_fraud_prob,con_fraud_prob
2,2021-08-20,15549624934,130.3505283105634,Commodo Associates,"opticians, optica...",c,2.76,NULL,NULL
18482,2021-08-20,46804135891,6.6168976971833615,Suspendisse Dui C...,"opticians, optica...",c,2.93,NULL,NULL
15,2021-08-20,11237511112,86.43306925925785,Magna Institute,"opticians, optica...",c,2.11,NULL,NULL
37,2021-08-20,48534649627,115.71011998439714,Dignissim Maecena...,"opticians, optica...",a,6.64,NULL,NULL
18563,2021-08-20,22059270846,13.552320313774313,Montes Nascetur R...,"opticians, optica...",a,6.59,NULL,NULL
83,2021-08-20,46804135891,40.88736687931136,Suspendisse Dui C...,"opticians, optica...",c,2.93,NULL,NULL
90,2021-08-20,81410315303,97.01139408070111,Sed Dictum PC,"opticians, optica...",a,6.35,NULL,NULL
96,2021-08-20,46804135891,71.58993600692456,Suspendisse Dui C...,"opticians, optica...",c,2.93,NULL,NULL
112,2021-08-20,60602272553,17.429130408212,Sagittis Duis Gra...,"opticians, optica...",b,4.93,NULL,NULL
118,2021-08-20,46804135891,14.417591756971596,Suspendisse Dui C...,"opticians, optica...",c,2.93,NULL,NULL


In [20]:
curated=mtc_fraud.groupBy(['merchant_abn', 'user_id']).agg(
    f.count('*').alias('count'),
    f.mean('dollar_value').alias('mean'),
    f.mean('merch_fraud_prob').alias('merch_fraud_prob'),
    f.mean('con_fraud_prob').alias('con_fraud_prob')
)



In [21]:
#curated

In [ ]:
new_curated=curated.join(consumer_user_details, on='user_id', how='left')
new_curated=new_curated.join(tbl_consumer, on='consumer_id', how='left')
new_curated=new_curated.drop('consumer_id')

In [29]:
new_curated.sort('con_fraud_prob', ascending=False).show()

25/10/05 13:49:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:49:28 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+------------+-----+------------------+------------------+-----------------+-----+--------+
|user_id|merchant_abn|count|              mean|  merch_fraud_prob|   con_fraud_prob|state|postcode|
+-------+------------+-----+------------------+------------------+-----------------+-----+--------+
|   6228| 21025433654|    5|15.216678564886047|              NULL| 97.6298077657765|  VIC|    3186|
|   8347| 73972053940|    4| 34.11222760222817|              NULL|92.99138306039121|  VIC|    3123|
|  16556| 49322182190|    5| 173.6878396697779|              NULL|89.65663294494827|  QLD|    4743|
|  16556| 92065881715|    2|192.82296700148154|              NULL|89.65663294494827|  QLD|    4743|
|   2310| 61412665910|    2| 263.2459528668431|              NULL|87.96418126148595|  VIC|    3677|
|   5233| 45629217853|   14| 34.64427256730033|              NULL|85.87123303878818|  VIC|    3597|
|   5233| 34695415993|    2| 87.75287320319775|              NULL|85.87123303878818|  VIC|    3597|


In [24]:
print(new_curated.shape)

25/10/05 13:22:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/05 13:22:41 WARN RowBasedKeyValueBatch: Calling spill() on

(7881926, 8)


In [25]:
#new_curated.write.parquet("data/curated/agg_by_userbiz", mode="overwrite")

In [26]:
#merchant_transactions.write.parquet("../data/curated/merchant_transactions", mode="overwrite")

In [27]:
dollar_vals=mtc_fraud.select("mean")

bin_edges, counts = (
    dollar_vals
    .rdd.flatMap(lambda x: x)     # flatten column
    .histogram(20)                # 20 bins
)

# Plot
plt.bar(
    bin_edges[:-1], counts,
    width=[bin_edges[i+1] - bin_edges[i] for i in range(len(bin_edges)-1)],
    align="edge", edgecolor="black"
)
plt.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `mean` cannot be resolved. Did you mean one of the following? [`rev_band`, `user_id`, `biz_tags`, `business`, `take_rate`].;
'Project ['mean]
+- Project [user_id#10L, order_datetime#14, merchant_abn#11L, dollar_value#12, business#873, biz_tags#198, rev_band#176, take_rate#184, merch_fraud_prob#2347, fraud_probability#59 AS con_fraud_prob#2368]
   +- Project [user_id#10L, order_datetime#14, merchant_abn#11L, dollar_value#12, business#873, biz_tags#198, rev_band#176, take_rate#184, merch_fraud_prob#2347, fraud_probability#59]
      +- Join LeftOuter, ((user_id#10L = cast(user_id#57 as bigint)) AND (order_datetime#14 = cast(order_datetime#58 as date)))
         :- Project [merchant_abn#11L, order_datetime#14, user_id#10L, dollar_value#12, business#873, biz_tags#198, rev_band#176, take_rate#184, fraud_probability#82 AS merch_fraud_prob#2347]
         :  +- Project [merchant_abn#11L, order_dateti